# BERT

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

In [21]:
data = pd.read_csv('../data/potential_talents_cleaned.csv')

In [58]:
# use basic pre-trained BERT model
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [60]:
# create a function to turn text into embeddings using the model
def get_sentence_embeddings(texts):

    # the tokeniser to turn the text into tokens in the the form of PyTorch tensors
    inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True
    )

    # don't train the model, only use it to turn the tokens into contextual embeddings for each token
    with torch.no_grad():
        outputs=model(**inputs)
    token_embeddings = outputs.last_hidden_state

    # create a mask that removes unwanted tokens (start, end, pad etc.)
    attention_mask = inputs['attention_mask']
    special_tokens_mask = torch.tensor([
        tokenizer.get_special_tokens_mask(
            ids.tolist(),
            already_has_special_tokens=True
        )
        for ids in inputs['input_ids']
    ])

    mask = attention_mask * (1 - special_tokens_mask)
    mask = mask.unsqueeze(-1)

    # pool embeddings into a single embedding for each job title, rather than each token
    sentence_embeddings = (
        (token_embeddings * mask).sum(dim=1)
        / mask.sum(dim=1).clamp(min=1)
    )

    return sentence_embeddings

In [70]:
# use funtion to create embeddings for the target and each candidate
target_title = "aspiring hr"
target_embeddings = get_sentence_embeddings([target_title])

candidate_titles = data['job_title'].to_list()
candidate_embeddings = get_sentence_embeddings(candidate_titles)

In [71]:
# assses the similarity of each candidate to the target using 4 different methods
cosine_similarity = F.cosine_similarity(
    target_embeddings,
    candidate_embeddings
)

dot_similarity = torch.matmul(
    target_embeddings, 
    candidate_embeddings.T
).squeeze(0)

euclidean_similarity = torch.cdist(
    target_embeddings,
    candidate_embeddings,
    p=2
).squeeze(0)

manhattan_similarity = torch.cdist(
    target_embeddings,
    candidate_embeddings,
    p=1
).squeeze(0)

In [72]:
# turn the similarity scores into ranks, sort by the cosine rank and show all ranks next to each other
data['cosine_rank'] = pd.Series(cosine_similarity.numpy(), index=data.index).rank(ascending=False, method='min').astype(int)
data['dot_rank'] = pd.Series(dot_similarity.numpy(), index=data.index).rank(ascending=False, method='min').astype(int)
data['euclidean_rank'] = pd.Series(euclidean_similarity.numpy(), index=data.index).rank(ascending=True, method='min').astype(int)
data['manhattan_rank'] = pd.Series(manhattan_similarity.numpy(), index=data.index).rank(ascending=True, method='min').astype(int)
data = data.sort_values(by='cosine_rank').reset_index(drop=True)

In [73]:
# display each candidates job title and 4 similarity ranks
data[['job_title', 'cosine_rank', 'dot_rank', 'euclidean_rank', 'manhattan_rank']]

,job_title,cosine_rank,dot_rank,euclidean_rank,manhattan_rank
0,aspiring hr specialist,1,1,1,1
1,aspiring hr professional,2,2,2,2
2,aspiring hr professional,2,2,2,2
3,hr professional,4,4,5,7
4,aspiring hr manager seeking internship in hr,5,9,4,4
5,hr senior specialist,6,7,7,9
6,hr management major,7,6,11,11
7,business management major and aspiring hr manager,8,8,8,6
8,aspiring hr management student seeking an inte...,9,10,6,5
9,experienced retail manager and aspiring hr pro...,10,11,9,8
